# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"The total revenue across all orders is ${total_revenue:,.2f}.")
print(f"The total number of units sold is {total_units:,}.")

The total revenue across all orders is $8,520.00.
The total number of units sold is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
cats = ['Food','Merch','RainGear','Drink']
category_revenues = df.groupby('category')['revenue'].sum()

for x in cats:

  cat_revenue_for_x = category_revenues.loc[x]
  percent_share = cat_revenue_for_x / total_revenue
  print(f"Revenue share for {x} is {percent_share:.2%}")

Revenue share for Food is 50.39%
Revenue share for Merch is 20.79%
Revenue share for RainGear is 10.58%
Revenue share for Drink is 18.24%


In [4]:
# Create `by_category` as a DataFrame for the assertion in Q7
by_category = category_revenues.to_frame(name='revenue')

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [5]:
vendor_ids = df['vendor_id'].unique()

order_revenue = df.groupby("vendor_id")["revenue"].sum()
order_count = df.groupby("vendor_id")["revenue"].count()

best = 0

for x in vendor_ids:
  cat_order_revenue = order_revenue.loc[x]
  cat_order_count = order_count.loc[x]
  average_order_revenue = cat_order_revenue / cat_order_count

  if average_order_revenue > best:
    best = average_order_revenue
    best_vendor = x

  print(f"Average order revenue for {x} is ${average_order_revenue:.2f}")

print(f"\nThe vendor with the highest average order revenue is {best_vendor} with ${best:.2f}")

Average order revenue for V-10 is $20.31
Average order revenue for V-18 is $21.75
Average order revenue for V-01 is $22.60
Average order revenue for V-05 is $20.58

The vendor with the highest average order revenue is V-01 with $22.60


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [6]:
cat_revenue_for_merch = category_revenues.loc["Merch"]
percent_share = cat_revenue_for_merch / total_revenue
print(f"Revenue share for merch is {percent_share:.1%}")

Revenue share for merch is 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [7]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# merge validate report

# store og counts
original_row_count = len(df)
original_total_revenue = df['revenue'].sum()

#  left merge
df_merged = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')

# validate
merged_row_count = len(df_merged)
merged_total_revenue = df_merged['revenue'].sum()

print(f"Original row count: {original_row_count}")
print(f"Merged row count: {merged_row_count}")
print(f"Row count unchanged: {original_row_count == merged_row_count}")

print(f"Original total revenue: ${original_total_revenue:,.2f}")
print(f"Merged total revenue: ${merged_total_revenue:,.2f}")
print(f"Total revenue unchanged: {abs(original_total_revenue - merged_total_revenue) < 0.01}")

unmatched_vendor_id = df_merged[df_merged['vendor_name'].isna()]['vendor_id'].unique()

# unmatched vendor
if len(unmatched_vendor_id) > 0:
    print(f"\nUnmatched vendor ID(s): {unmatched_vendor_id}")

else:
    print("\nAll vendor IDs matched.")

df = df_merged.copy()

Original row count: 400
Merged row count: 400
Row count unchanged: True
Original total revenue: $8,520.00
Merged total revenue: $8,520.00
Total revenue unchanged: True

Unmatched vendor ID(s): ['V-18']


In [8]:
# Define `joined` for the assertion in Q7
joined = df

**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [9]:
pivot_table_revenue = pd.pivot_table(df,
                                     index='vendor_name',
                                     columns='category',
                                     values='revenue',
                                     aggfunc='sum',
                                     margins=True) # row and column totals

print("Revenue by Vendor and Category:")
# fill na will add NaN when no data
# apply map formatting function
print(pivot_table_revenue.fillna(0).map(lambda x: f'${x:,.2f}'))

Revenue by Vendor and Category:
category           Drink       Food      Merch RainGear        All
vendor_name                                                       
Cav Merch North  $502.50  $1,054.50    $400.50  $175.50  $2,133.00
Hoos Burgers     $171.00  $1,338.00    $373.50  $241.50  $2,124.00
Rotunda Tacos    $298.50    $882.00    $489.00  $244.50  $1,914.00
All              $972.00  $3,274.50  $1,263.00  $661.50  $6,171.00


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [10]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(category_revenues.sum() - df['revenue'].sum()) < 0.01
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Considering how food accounts for 50.39% of the total revenue, I would advise vendors to focus on food sales as they seemed to make up the largest portion of profits, meanwhile categories such as raingear fall heavily behind (at 10.58%), so focusing on those is not the most efficient strategy.

b) The least trustworthy answer for the revenue breakdown would likely be V-18, as when we joined in the vendor names via. left join, we could see that one vendor ID in the orders was not in the lookup and did not have a vendor name. Since the data is incomplete, this could potentially
